# DSTS — LLM Inference Optimisation: 8B Model Benchmark (Pruning Only)

This notebook runs the **vocabulary-pruning** experiments on `meta-llama/Llama-3.1-8B-Instruct`
on a T4 x2 GPU. Speculative-decoding scripts are omitted (negative result on 1B).

## Prerequisites (do these ONCE before running)

### 1. HuggingFace token (for Llama-3.1-8B-Instruct access)
1. Go to https://huggingface.co/settings/tokens → create a token with **Read** scope
2. Accept the **Llama-3.1** license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
   (separate gate from Llama-3.2 — must be accepted even if you already accepted 3.2)
3. In Kaggle: top-right avatar → **Settings** → **Secrets** → **Add a new secret**
   - Name: `HF_TOKEN`
   - Value: your HuggingFace token (starts with `hf_...`)
4. In this notebook: click the **🔑 Add-ons** menu → **Secrets** → toggle `HF_TOKEN` to **Attached**

### 2. GitHub token (to clone the private repo)
Same as the 1B notebook — `GITHUB_TOKEN` secret must be attached.

### 3. GPU runtime — MUST be T4 x2
- Notebook Settings → Accelerator → **GPU T4 x2**
- `Llama-3.1-8B-Instruct` in bfloat16 requires ~16 GB VRAM; T4 x2 (2×16 GB = 32 GB) is needed
  for KV-cache and router overhead headroom.

### 4. No pretrained artefacts needed
The `router_checkpoint.pt` and `mlp_transition_graph.pt` from the 1B run are **incompatible**
with the 8B model (different hidden dimension: 2048 → 4096). Both will be rebuilt from scratch.
This adds ~60 min to the dual-encoder + graph experiments.

---
**After running:** go to the notebook's **Output** tab → download `results.zip`  
**Locally:** unzip into your `dsts/results_8b/` folder, then run `python run_plots.py`

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — check accelerator settings')

import torch
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'CUDA device {i}: {torch.cuda.get_device_name(i)}, '
              f'{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: HuggingFace token + model selection ───────────────────────────────
# Sets HF_TOKEN from Kaggle Secrets and selects the 8B model via MODEL_NAME env var.
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # legacy env var
    print('HF_TOKEN loaded from Kaggle Secrets ✓')
except Exception as e:
    print(f'WARNING: Could not load HF_TOKEN from Kaggle Secrets: {e}')
    raise

# Select the 8B model — all run_*.py scripts read MODEL_NAME from the environment
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
print(f'MODEL_NAME set to: {os.environ["MODEL_NAME"]}')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
# Kaggle already has torch/transformers; we pin versions and add missing packages.
!pip install -q \
    'transformers>=4.40.0' \
    'accelerate>=0.29.0' \
    'datasets>=2.18.0' \
    'sentencepiece>=0.2.0' \
    'tokenizers>=0.19.0' \
    'scikit-learn>=1.4.0' \
    'sentence-transformers>=2.7.0' \
    'tqdm>=4.66.0' \
    'numpy>=1.26.0' \
    'pandas>=2.2.0' \
    'matplotlib>=3.8.0' \
    'seaborn>=0.13.0'

print('Dependencies installed ✓')

In [ ]:
# ── Cell 4: Clone repo (private) ─────────────────────────────────────────────
# Uses a GitHub Personal Access Token (PAT) stored in Kaggle Secrets.
import os
from kaggle_secrets import UserSecretsClient

gh_token = UserSecretsClient().get_secret('GITHUB_TOKEN')

WORKDIR = '/kaggle/working/dsts'
REPO_URL = f'https://{gh_token}@github.com/deil87/dsts.git'

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {WORKDIR} remote set-url origin {REPO_URL}
    !git -C {WORKDIR} pull

%cd {WORKDIR}
print(f'Working directory: {os.getcwd()}')

# Re-export MODEL_NAME after %cd (subshells inherit env from the kernel)
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
print(f'MODEL_NAME: {os.environ["MODEL_NAME"]}')
!ls -la

In [ ]:
# ── Cell 5: Artefacts note ────────────────────────────────────────────────────
# The 1B router_checkpoint.pt (hidden_dim=2048) and mlp_transition_graph.pt are
# INCOMPATIBLE with Llama-3.1-8B-Instruct (hidden_dim=4096). They will be
# rebuilt from scratch during run_dual_encoder.py (~50 min) and run_graph.py (~15 min).
#
# If you have 8B-compatible artefacts in a Kaggle Dataset named 'dsts-artefacts-8b',
# attach it and uncomment the copy block below.

import os
os.makedirs('results', exist_ok=True)

import subprocess
DATASET_ROOT = None
result = subprocess.run(
    ['find', '/kaggle/input', '-name', 'router_checkpoint.pt'],
    capture_output=True, text=True
)
if result.stdout.strip():
    DATASET_ROOT = os.path.dirname(result.stdout.strip().splitlines()[0])
    print(f'Found artefacts at: {DATASET_ROOT}')
    print('WARNING: These artefacts may be from the 1B run (hidden_dim=2048).')
    print('They will be IGNORED — rebuilding from scratch for 8B (hidden_dim=4096).')
else:
    print('No pretrained artefacts found — will build from scratch (expected).')
    print('  run_dual_encoder.py will train RouterMLP (~50 min)')
    print('  run_graph.py will build MLP transition graph (~15 min)')

In [ ]:
# ── Cell 6: Run baseline ──────────────────────────────────────────────────────
# Measures: tok/s, lm_head fraction, baseline PPL
# Expected runtime: ~15 min (8B on T4x2)
print('=' * 60)
print('EXPERIMENT 1/6: Baseline')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_baseline.py

In [ ]:
# ── Cell 7: Static + Cosine router ───────────────────────────────────────────
# Vocabulary pruning with static weight-norm ranking and cosine similarity.
# Expected runtime: ~40 min (8B on T4x2)
print('=' * 60)
print('EXPERIMENT 2/6: Static + Cosine Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_static.py

In [ ]:
# ── Cell 8: Cluster router ────────────────────────────────────────────────────
# k-means cluster-based vocabulary pruning.
# Expected runtime: ~25 min (8B on T4x2)
print('=' * 60)
print('EXPERIMENT 3/6: Cluster Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_cluster.py

In [ ]:
# ── Cell 9: Dual-encoder router ───────────────────────────────────────────────
# Trains RouterMLP from scratch (no compatible 1B checkpoint) then runs
# step-level + prefetch/refresh dual-encoder routing.
# Expected runtime: ~90 min (8B on T4x2, includes ~50 min training)
print('=' * 60)
print('EXPERIMENT 4/6: Dual-Encoder Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_dual_encoder.py

In [ ]:
# ── Cell 10: MLP transition graph router ─────────────────────────────────────
# MLP up/down projection co-activation graph. Rebuilt from scratch for 8B.
# Expected runtime: ~45 min (8B on T4x2, includes ~15 min graph build)
print('=' * 60)
print('EXPERIMENT 5/6: MLP Transition Graph Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_graph.py

In [ ]:
# ── Cell 11: Plots ────────────────────────────────────────────────────────────
# Generates all summary figures from the CSV results.
# NOTE: run_attention.py is skipped (output_attentions=True overhead too high on 8B).
# NOTE: run_speculative*.py are skipped (negative results established on 1B).
# Expected runtime: ~1 min
print('=' * 60)
print('EXPERIMENT 6/6: Generate Plots')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_plots.py

In [ ]:
# ── Cell 12: Summary of results ───────────────────────────────────────────────
import pandas as pd
import os

print('Results directory contents:')
for f in sorted(os.listdir('results')):
    size = os.path.getsize(f'results/{f}')
    print(f'  {f:<50s} {size/1024:>8.1f} KB')

# Print the consolidated results table if it exists
all_csv = 'results/all_results.csv'
if os.path.exists(all_csv):
    print('\n=== All Results ===')
    df = pd.read_csv(all_csv)
    with pd.option_context('display.max_rows', 50, 'display.max_columns', 20,
                           'display.width', 120, 'display.float_format', '{:.3f}'.format):
        print(df.to_string(index=False))

In [ ]:
# ── Cell 13: Package results for download ────────────────────────────────────
# Zips the entire results/ directory → /kaggle/working/results.zip
# Download it from the Output tab on the right side of this notebook page.
import shutil, os

OUTPUT_ZIP = '/kaggle/working/results'
shutil.make_archive(OUTPUT_ZIP, 'zip', 'results')

zip_size = os.path.getsize(OUTPUT_ZIP + '.zip') / 1e6
print(f'results.zip created: {zip_size:.1f} MB')
print()
print('To download:')
print('  1. Click the Output tab (right panel of this notebook)')
print('  2. Find results.zip and click the download icon')
print()
print('Locally, unzip into your dsts/results_8b/ folder:')
print('  cd /path/to/dsts')
print('  mkdir -p results_8b')
print('  unzip ~/Downloads/results.zip -d results_8b/')
print('  RESULTS_DIR=results_8b python run_plots.py   # if run_plots supports RESULTS_DIR')